# Phase 1: Watching optimizers move on low-dimensional surfaces

In this notebook we optimize a single 2-D point over a few classic loss surfaces and **watch the path** each optimizer takes. The goal is intuition, not realism:

- See how **SGD**, **momentum**, **AdaGrad**, and **Adam** differ.
- Build a feel for convergence speed, stability, and sensitivity to the learning rate.

The optimizers here are written from scratch in [`workshoplib/optimizers.py`](../workshoplib/optimizers.py) so you can read the exact update rules. (MuON is introduced later, in Phase 2, where the network has real weight *matrices* for it to act on.)

In [ ]:
from pathlib import Path
import os
import sys

# Make sure we run from the project root so 'import workshoplib' works.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    os.chdir(ROOT.parent)
    ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Working directory:", ROOT)

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import matplotlib.pyplot as plt

from workshoplib.objectives import OBJECTIVES, get_objective
from workshoplib.optimizers import run_descent
from workshoplib import viz

OPTIMIZERS = ["sgd", "momentum", "adagrad", "adam"]
print("Available objectives:", list(OBJECTIVES))

## A loss surface and a gradient step

Each objective is a function that maps a 2-D point `(x, y)` to a single number (the *loss*). We can picture it as a landscape: the height is the loss, and optimization is the process of walking downhill.

Every optimizer here does the same basic thing each step: compute the gradient (the steepest-uphill direction) and move in the opposite direction. They differ in **how** they use that gradient (raw, smoothed with momentum, or rescaled per-coordinate).

The helper below runs all four optimizers from the same starting point and shows two views: the **path on the contour map** and the **loss vs. step** curve.

In [ ]:
def compare(key, n_steps=80, optim_selected=OPTIMIZERS):
    """Run every optimizer on one objective and show both diagnostic plots."""
    objective = get_objective(key)

    trajectories = [run_descent(objective, name, n_steps=n_steps) for name in optim_selected]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    viz.plot_trajectory_on_contour(objective, trajectories, ax=axes[0])
    viz.plot_loss_curves(trajectories, ax=axes[1])
    fig.tight_layout()
    plt.show()
    return trajectories

## 1. Quadratic bowl (the easy case)

`f = x^2 + y^2`. A symmetric bowl with its minimum at the origin. Here almost everything works: it is a good sanity check and baseline before we make life harder.

In [ ]:
_ = compare("quadratic_bowl", n_steps=80)

**What to notice:** all four methods head straight for the center. Differences are small because the surface is perfectly conditioned (equally curved in every direction).

## 2. Ill-conditioned quadratic (the stretched bowl)

`f = x^2 + 25 y^2`. Now the bowl is 25x steeper in `y` than in `x`. A single learning rate is a compromise: large enough to make progress in the shallow `x` direction tends to overshoot and oscillate in the steep `y` direction.

In [ ]:
_ = compare("ill_conditioned", n_steps=10, optim_selected=["sgd","adagrad"])

**What to notice:** plain SGD zig-zags down the steep walls and crawls along the shallow valley floor. Momentum smooths the zig-zag; AdaGrad and Adam rescale each coordinate's step and reach the center far more directly. This per-coordinate scaling is the whole point of adaptive methods.

## 3. Rotated ill-conditioned quadratic (off-axis directions)

`f = (x + y)^2 + 25 (x - y)^2`. Same idea as the previous bowl - a 25:1 stretch (curvature eigenvalues 4 and 100) - but rotated 45 degrees: the gentle direction now runs along the diagonal `x = y` and the steep direction runs across it. Only the **orientation** of the valley changed.

AdaGrad and Adam rescale each coordinate (`x` and `y`) *independently*. That trick only matches the geometry when the steep and shallow directions happen to be the coordinate axes. Watch what happens when they are not.

In [ ]:
_ = compare("rotated_ill_conditioned", n_steps=50, optim_selected=["sgd","adagrad"])

**What to notice:** on the *axis-aligned* bowl (section 2) AdaGrad reached the minimum fastest. Here, with the very same curvature but rotated, AdaGrad is now among the *slowest* - its per-coordinate scaling no longer lines up with the true steep and shallow directions, so it does little better than plain SGD. Momentum acts along the actual gradient direction rather than per-coordinate, so the rotation barely affects it. This is the key limitation of diagonal adaptive methods: they are biased toward the coordinate axes.

## 4. Absolute-value valley (momentum, simply)

`f = |x| + |y|`. A sharp V-shaped valley meeting at the minimum `(0, 0)`. The gradient is piecewise constant - magnitude 1 along each axis, always pointing toward 0. Starting from the far left, the **x-component of the gradient points right at every single step**, while the y-component just flips sign as the path crosses the valley floor.

This is about the simplest setting in which to see what momentum buys you, with no curvature to reason about.

In [ ]:
_ = compare("abs_valley", n_steps=70, optim_selected=["sgd","momentum"])

**What to notice:** plain SGD inches right at a constant speed - one fixed step per iteration - and is still well left of the minimum after 30 steps, jittering up and down by one step size as it crosses the valley floor. Momentum accumulates the steady rightward pull into a growing velocity and reaches the minimum region in about a third of the steps; watch how much faster its loss curve drops early on. Because the minimum is a sharp kink, momentum then overshoots and oscillates around it - the same accumulated speed that gets it there fast also makes it harder to stop. (Adam carries a velocity-like term too, so it accelerates similarly.)

## 5. Rosenbrock (the curved valley)

`f = (1 - x)^2 + 100 (y - x^2)^2`. The famous "banana". The minimum at `(1, 1)` sits at the bottom of a long, curved, narrow valley. It is easy to drop into the valley but hard to follow it to the bottom, so we give it more steps.

In [ ]:
_ = compare("rosenbrock", n_steps=800, optim_selected=["sgd","momentum"])

**What to notice:** plain SGD and AdaGrad stall partway along the valley. Momentum and Adam carry speed around the bend and make much more progress toward `(1, 1)`. This is where momentum really earns its keep.

## 6. Saddle point

`f = x^2 - y^2`. There is **no minimum**: the origin is a saddle (a minimum along `x`, a maximum along `y`). Optimizers should slide down along `y` and away. This illustrates that not every flat-looking point is a good place to stop.

In [ ]:
_ = compare("saddle", n_steps=80)

**What to notice:** the loss keeps decreasing (it can go negative here) as the optimizers escape along the downhill `y` direction. Methods that build up speed or rescale steps leave the saddle's flat ridge faster than plain SGD.

## 7. Beale (optional, harder)

A multi-feature surface with broad flat regions and sharp valleys, minimum at `(3, 0.5)`. A good stress test for keen students.

In [ ]:
_ = compare("beale", n_steps=300)

**What to notice:** the flat regions starve plain SGD of gradient, so it barely moves; momentum and Adam make far more progress toward the minimum.

## 8. Learning-rate sensitivity

The learning rate is the single most important knob. Too small and training crawls; too large and it oscillates or diverges. Below we run plain **SGD** on the rotated ill-conditioned quadratic at several learning rates and compare the loss curves.

The steep direction has curvature 100, so SGD is only stable for learning rates below about `2 / 100 = 0.02`. Watch what happens as we cross that threshold.

In [ ]:
objective_key = "rotated_ill_conditioned" # OBJECTIVES = ['quadratic_bowl', 'ill_conditioned', 'rotated_ill_conditioned', 'rosenbrock', 'saddle', 'beale']
algo_key = "sgd" # OPTIMIZERS = ['sgd', 'momentum', 'adagrad', 'adam']
objective = get_objective(objective_key)
learning_rates = [0.021, 0.02,0.01,0.001]

fig, ax = plt.subplots(figsize=(7, 5))
for lr in learning_rates:
    traj = run_descent(objective, algo_key, lr=lr, n_steps=50)
    ax.plot(traj.losses.numpy(), label=f"lr = {lr}")

ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_title("learning-rate sensitivity: %s :  %s" % (objective_key, algo_key))
ax.legend()
plt.show()

**What to notice:** tiny rates converge slowly; a well-chosen rate converges fast; a rate past the stability limit makes the loss grow instead of shrink.

## Try it yourself

- Change the starting point: `run_descent(get_objective("rosenbrock"), "adam", x0=(1.5, -0.5), n_steps=300)`.
- Override learning rates per optimizer and see when each diverges.
- Add your own 2-D objective to [`workshoplib/objectives.py`](../workshoplib/objectives.py) and rerun `compare("your_key")`.
- Visualize a surface in 3-D: `viz.plot_surface_3d(get_objective("rosenbrock")); plt.show()`.